# Yahoo and Polygon Options Example

This notebook shows two common option-data workflows for a single underlying:

- Yahoo Finance option chain discovery with `yfinance`
- Polygon contract discovery and single-contract snapshot lookup

Default underlying: `AAPL`

Notes:
- Yahoo can be rate-limited or return partial option data.
- Polygon requires `POLYGON_API_KEY` in your environment.


## Setup

In PowerShell before starting Jupyter:

```powershell
$env:POLYGON_API_KEY = "your-key-here"
```

If `yfinance` is missing in your notebook kernel:

```powershell
pip install yfinance pandas requests
```


In [1]:
import os
from datetime import datetime

import pandas as pd
import requests
import yfinance as yf

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

UNDERLYING = "SPY"
POLYGON_API_KEY = "eR70Jqcd8JhQ5MTZmDp_YKpXMndaNVrk"  #os.getenv("POLYGON_API_KEY")

# Yahoo Finance symbols for option chains (index options use different tickers than price feeds)
_YF_TICKER_MAP = {
    "SPX": "^SPX",    # CBOE SPX options  (^GSPC is price-only, no chain)
    "NDX": "^NDX",
    "RUT": "^RUT",
    "VIX": "^VIX",
    "DJX": "^DJI",
}
_yf_symbol = _YF_TICKER_MAP.get(UNDERLYING, UNDERLYING)

# Separate price-only map — ^GSPC has reliable price but no options chain
_YF_PRICE_MAP = {
    "SPX": "^GSPC",
    "NDX": "^NDX",
    "RUT": "^RUT",
    "DJX": "^DJI",
}
_yf_price_symbol = _YF_PRICE_MAP.get(UNDERLYING, UNDERLYING)

print({
    "underlying": UNDERLYING,
    "yf_options_symbol": _yf_symbol,
    "yf_price_symbol": _yf_price_symbol,
    "polygon_key_present": bool(POLYGON_API_KEY),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
})


{'underlying': 'SPY', 'yf_options_symbol': 'SPY', 'yf_price_symbol': 'SPY', 'polygon_key_present': True, 'timestamp': '2026-08-22T07:33:27'}


## Yahoo: discover expirations and option chain


In [2]:
ticker = yf.Ticker(_yf_symbol)

try:
    expirations = list(ticker.options)
except Exception as exc:
    raise RuntimeError(
        f"Yahoo option discovery failed for {_yf_symbol}. This usually means rate limiting or an upstream Yahoo response issue."
    ) from exc

if not expirations:
    raise RuntimeError(
        f"Yahoo returned no option expirations for {_yf_symbol}. This often means rate limiting or missing upstream data."
    )

expiration_df = pd.DataFrame({"expiration": expirations})
expiration_df.head(10)


,expiration
0,2026-08-24
1,2026-08-25
2,2026-08-26
3,2026-08-27
4,2026-08-28
5,2026-08-31
6,2026-09-04
7,2026-09-11
8,2026-09-18
9,2026-09-25


In [ ]:
pd.head(10)

In [3]:
selected_expiration = expirations[9]

try:
    chain = ticker.option_chain(selected_expiration)
except Exception as exc:
    raise RuntimeError(
        f"Yahoo option chain lookup failed for expiration {selected_expiration}."
    ) from exc

calls = chain.calls.copy()
puts = chain.puts.copy()

if calls.empty and puts.empty:
    raise RuntimeError("Yahoo returned an empty option chain for the selected expiration.")

print("selected_expiration:", selected_expiration)
print("calls:", len(calls), "puts:", len(puts))

calls.head(10)


selected_expiration: 2026-09-25
calls: 120 puts: 117


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,SPY260925C00550000,2026-08-13 17:39:55+00:00,550.0,229.19,215.46,218.98,0.0,0.0,NaN,1.0,0.580326,True,REGULAR,USD
1,SPY260925C00590000,2026-08-07 14:57:31+00:00,590.0,185.40,175.66,179.18,0.0,0.0,2.0,NaN,0.562749,True,REGULAR,USD
2,SPY260925C00600000,2026-08-20 14:13:15+00:00,600.0,168.55,165.73,169.25,0.0,0.0,1.0,259.0,0.535954,True,REGULAR,USD
3,SPY260925C00640000,2026-08-18 13:45:28+00:00,640.0,131.30,126.03,129.55,0.0,0.0,9.0,766.0,0.429815,True,REGULAR,USD
4,SPY260925C00645000,2026-08-13 15:01:02+00:00,645.0,136.18,121.08,124.60,0.0,0.0,NaN,1.0,0.416815,True,REGULAR,USD
5,SPY260925C00660000,2026-08-10 16:40:55+00:00,660.0,116.20,106.26,109.78,0.0,0.0,NaN,1.0,0.378424,True,REGULAR,USD
6,SPY260925C00670000,2026-08-12 17:03:20+00:00,670.0,106.58,96.40,99.92,0.0,0.0,NaN,17.0,0.352942,True,REGULAR,USD
7,SPY260925C00675000,2026-08-19 15:54:46+00:00,675.0,98.38,91.48,95.00,0.0,0.0,1.0,2.0,0.340278,True,REGULAR,USD
8,SPY260925C00680000,2026-08-19 15:37:32+00:00,680.0,93.80,86.58,90.10,0.0,0.0,1.0,12.0,0.327949,True,REGULAR,USD
9,SPY260925C00685000,2026-08-18 14:50:29+00:00,685.0,87.12,81.69,85.21,0.0,0.0,1.0,3.0,0.315620,True,REGULAR,USD


In [31]:
import math
from datetime import date as _date

_DELTA_TARGET  = 0.20
_DELTA_TOL     = 0.05   # keep |delta| in [0.15, 0.25]
_IV_FALLBACK   = 0.20
_RISK_FREE     = 0.05


def _ncdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2))


def _bs_delta(S, K, T, sigma, option_type, rf=_RISK_FREE):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (rf + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return _ncdf(d1) if option_type == "call" else _ncdf(d1) - 1.0


# underlying price
_pu = yf.Ticker(_yf_price_symbol)
_pl = getattr(_pu.fast_info, "last_price", None)
if not _pl:
    _ph = _pu.history(period="5d")["Close"].dropna()
    if _ph.empty:
        raise RuntimeError(f"Could not fetch price for {_yf_price_symbol}")
    _pl = float(_ph.iloc[-1])
_S = float(_pl)

_dte_yf = max((_date.fromisoformat(selected_expiration) - _date.today()).days, 1)
_T = _dte_yf / 365.0
print(f"underlying: {_S:.2f}   expiration: {selected_expiration}   dte: {_dte_yf}")

# compute delta for each row
def _add_delta(df, option_type):
    df = df.copy()
    df["delta"] = df.apply(
        lambda row: _bs_delta(_S, row["strike"], _T, row["impliedVolatility"] or _IV_FALLBACK, option_type),
        axis=1,
    ).round(4)
    return df

_calls_all = _add_delta(calls, "call")
_puts_all  = _add_delta(puts,  "put")

# filter: call delta ≈ +0.20, put |delta| ≈ 0.20
calls_d20 = _calls_all[(_calls_all["delta"] - _DELTA_TARGET).abs() <= _DELTA_TOL].reset_index(drop=True)
puts_d20  = _puts_all[(_puts_all["delta"].abs() - _DELTA_TARGET).abs() <= _DELTA_TOL].reset_index(drop=True)

print(f"calls with delta ≈ {_DELTA_TARGET} (±{_DELTA_TOL}): {len(calls_d20)}")
display(calls_d20[["contractSymbol", "strike", "delta", "bid", "ask", "impliedVolatility", "openInterest"]])

print(f"\nputs  with |delta| ≈ {_DELTA_TARGET} (±{_DELTA_TOL}): {len(puts_d20)}")
display(puts_d20[["contractSymbol", "strike", "delta", "bid", "ask", "impliedVolatility", "openInterest"]])


underlying: 720.65   expiration: 2026-05-15   dte: 12
calls with delta ≈ 0.2 (±0.05): 5


,contractSymbol,strike,delta,bid,ask,impliedVolatility,openInterest
0,SPY260515C00734000,734.0,0.2301,2.04,2.06,0.122934,1640
1,SPY260515C00735000,735.0,0.2095,1.78,1.81,0.121652,9638
2,SPY260515C00736000,736.0,0.1896,1.55,1.58,0.120401,1006
3,SPY260515C00737000,737.0,0.1705,1.35,1.37,0.119088,1423
4,SPY260515C00738000,738.0,0.1524,1.16,1.18,0.117807,501



puts  with |delta| ≈ 0.2 (±0.05): 9


,contractSymbol,strike,delta,bid,ask,impliedVolatility,openInterest
0,SPY260515P00700000,700.0,-0.1549,2.21,2.24,0.169381,51236
1,SPY260515P00701000,701.0,-0.1639,2.34,2.37,0.167672,1879
2,SPY260515P00702000,702.0,-0.1731,2.48,2.50,0.165780,1969
3,SPY260515P00703000,703.0,-0.1833,2.62,2.65,0.164193,2728
4,SPY260515P00704000,704.0,-0.1937,2.77,2.80,0.162392,5499
5,SPY260515P00705000,705.0,-0.2047,2.94,2.96,0.160592,23237
6,SPY260515P00706000,706.0,-0.2166,3.11,3.14,0.159036,2852
7,SPY260515P00707000,707.0,-0.2288,3.29,3.32,0.157235,3341
8,SPY260515P00708000,708.0,-0.2417,3.49,3.51,0.155404,3098


In [ ]:
puts.head(10)


## Yahoo: pick one contract and inspect it

Yahoo's chain already includes `contractSymbol`, which is usually the easiest contract identifier to reuse.


In [25]:
sample_call = calls.sort_values("strike").iloc[0]
yahoo_contract_symbol = sample_call["contractSymbol"]

sample_call[[
    "contractSymbol",
    "strike",
    "lastPrice",
    "bid",
    "ask",
    "impliedVolatility",
    "openInterest",
    "volume",
]].to_frame(name="value")


,value
contractSymbol,SPX260515C00200000
strike,200.0
lastPrice,6915.17
bid,7004.2
ask,7028.2
impliedVolatility,0.00001
openInterest,16
volume,1.0


In [26]:
option_ticker = yf.Ticker(yahoo_contract_symbol)
option_history = option_ticker.history(period="5d", interval="1d")

print("yahoo_contract_symbol:", yahoo_contract_symbol)
print("history_rows:", len(option_history))
option_history.tail()


yahoo_contract_symbol: SPX260515C00200000
history_rows: 1


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-04-29 00:00:00-04:00,6915.169922,6915.169922,6915.169922,6915.169922,1,0.0,0.0


## Polygon helpers


In [7]:
if not POLYGON_API_KEY:
    raise RuntimeError("Set POLYGON_API_KEY in your environment before running Polygon examples.")

POLYGON_BASE_URL = "https://api.polygon.io"
os.environ["POLYGON_API_KEY"] = "eRpXsk9YhoXF1OiKEk2xlpsFrhZoAkh0"

def polygon_get(path: str, **params):
    response = requests.get(
        f"{POLYGON_BASE_URL}{path}",
        params={**params, "apiKey": POLYGON_API_KEY},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


## Polygon: discover contracts for the same underlying and expiration

This uses Polygon's `GET /v3/reference/options/contracts` endpoint.


In [10]:
from datetime import date
os.environ["POLYGON_API_KEY"] = "eRpXsk9YhoXF1OiKEk2xlpsFrhZoAkh0"

# Fetch distinct expirations from Polygon — limit=1000 and gte today to avoid buried results
_exp_json = polygon_get(
    "/v3/reference/options/contracts",
    underlying_ticker=UNDERLYING,
    contract_type="call",
    **{"expiration_date.gte": date.today().isoformat()},
    order="asc",
    sort="expiration_date",
    limit=1000,
)
_exp_results = _exp_json.get("results", [])
if not _exp_results:
    raise RuntimeError("Polygon returned no contracts for this underlying. Check UNDERLYING and POLYGON_API_KEY.")

polygon_expirations = sorted({r["expiration_date"] for r in _exp_results})
selected_expiration = polygon_expirations[min(9, len(polygon_expirations) - 1)]

print(f"found {len(polygon_expirations)} expirations:")
for exp in polygon_expirations:
    marker = " <-- selected" if exp == selected_expiration else ""
    print(f"  {exp}{marker}")


HTTPError: 401 Client Error: Unauthorized for url: https://api.polygon.io/v3/reference/options/contracts?underlying_ticker=SPY&contract_type=call&expiration_date.gte=2026-08-22&order=asc&sort=expiration_date&limit=1000&apiKey=eR70Jqcd8JhQ5MTZmDp_YKpXMndaNVrk

import requests

url = "https://api.massive.com/stocks/filings/vX/index"
params = {
    "ticker": "AAPL",
    "form_type": "10-K",
    "limit": 5,
    "sort": "filing_date.desc"
}

response = requests.get(url, params=params, headers={"Authorization": f"Bearer {API_KEY}"})
filings = response.json()["results"]

for filing in filings:
    print(f"{filing['filing_date']}  {filing['form_type']}  {filing['issuer_name']}")
    print(f"  {filing['filing_url']}")

In [12]:
import requests
API_KEY = "eRpXsk9YhoXF1OiKEk2xlpsFrhZoAkh0"
url = "https://api.massive.com/stocks/filings/vX/index"
params = {
    "ticker": "AAPL",
    "form_type": "10-K",
    "limit": 5,
    "sort": "filing_date.desc"
}

response = requests.get(url, params=params, headers={"Authorization": f"Bearer {API_KEY}"})
filings = response.json()["results"]

for filing in filings:
    print(f"{filing['filing_date']}  {filing['form_type']}  {filing['issuer_name']}")
    print(f"  {filing['filing_url']}")

2025-10-31  10-K  Apple Inc.
  https://www.sec.gov/Archives/edgar/data/320193/0000320193-25-000079.txt
2024-11-01  10-K  Apple Inc.
  https://www.sec.gov/Archives/edgar/data/320193/0000320193-24-000123.txt
2023-11-03  10-K  Apple Inc.
  https://www.sec.gov/Archives/edgar/data/320193/0000320193-23-000106.txt
2022-10-28  10-K  Apple Inc.
  https://www.sec.gov/Archives/edgar/data/320193/0000320193-22-000108.txt
2021-10-29  10-K  Apple Inc.
  https://www.sec.gov/Archives/edgar/data/320193/0000320193-21-000105.txt


In [24]:
import yfinance as yf

ticker = yf.Ticker("nvda")

# Get current price info
# info = ticker.info
# print("Company:", info.get("longName"))
# print("Current price:", info.get("currentPrice"))

# Get historical data
hist = ticker.history(period="3mo")
print(hist)
# ticker.option_chain
print("Options expirations:", ticker.option_chain())

                                 Open        High         Low       Close     Volume  Dividends  Stock Splits
Date                                                                                                         
2026-05-22 00:00:00-04:00  220.642838  220.752711  214.549949  215.079330  169275700        0.0           0.0
2026-05-26 00:00:00-04:00  216.287916  217.926006  211.753207  214.609879  187202600        0.0           0.0
2026-05-27 00:00:00-04:00  213.870728  213.900692  208.536948  212.352509  167601200        0.0           0.0
2026-05-28 00:00:00-04:00  211.034036  215.269106  210.974108  214.000580  143996000        0.0           0.0
2026-05-29 00:00:00-04:00  214.330209  217.606389  210.884228  210.894211  289410600        0.0           0.0
...                               ...         ...         ...         ...        ...        ...           ...
2026-08-17 00:00:00-04:00  225.979996  227.919998  224.860001  225.009995   93678700        0.0           0.0
2026-08-18

In [29]:
import sys
import yfinance as yf

print("Python:", sys.executable)
print("yfinance:", yf.__version__)
print("yfinance module:", yf.__file__)

Python: c:\Python312\python.exe
yfinance: 1.2.1
yfinance module: C:\Users\zhaih\AppData\Roaming\Python\Python312\site-packages\yfinance\__init__.py


In [30]:
ticker = yf.Ticker("NVDA")

for label, request in {
    "fast_info": lambda: dict(ticker.fast_info),
    "history_metadata": ticker.get_history_metadata,
}.items():
    try:
        result = request()
        if label == "fast_info":
            result = {
                key: result.get(key)
                for key in ("last_price", "previous_close", "day_high", "day_low")
            }
        print(f"{label}: OK", result)
    except Exception as exc:
        print(f"{label}: FAILED - {type(exc).__name__}: {exc}")

fast_info: OK {'last_price': None, 'previous_close': None, 'day_high': None, 'day_low': None}
history_metadata: OK {'currency': 'USD', 'symbol': 'NVDA', 'exchangeName': 'NMS', 'fullExchangeName': 'NasdaqGS', 'instrumentType': 'EQUITY', 'firstTradeDate': 917015400, 'regularMarketTime': 1787342400, 'hasPrePostMarketData': True, 'gmtoffset': -14400, 'timezone': 'EDT', 'exchangeTimezoneName': 'America/New_York', 'regularMarketPrice': 214.72, 'fiftyTwoWeekHigh': 236.54, 'fiftyTwoWeekLow': 164.07, 'regularMarketDayHigh': 218.74, 'regularMarketDayLow': 214.5, 'regularMarketVolume': 91591112, 'longName': 'NVIDIA Corporation', 'shortName': 'NVIDIA Corporation', 'chartPreviousClose': 225.16, 'previousClose': 216.85, 'scale': 3, 'priceHint': 2, 'currentTradingPeriod': {'pre': {'timezone': 'EDT', 'start': 1787299200, 'end': 1787319000, 'gmtoffset': -14400}, 'regular': {'timezone': 'EDT', 'start': 1787319000, 'end': 1787342400, 'gmtoffset': -14400}, 'post': {'timezone': 'EDT', 'start': 1787342400, 

In [31]:
fast_info = dict(yf.Ticker("NVDA").fast_info)
print("fast_info keys:", sorted(fast_info))
print(
    "fast_info quote:",
    {
        key: fast_info.get(key)
        for key in ("lastPrice", "previousClose", "dayHigh", "dayLow")
    },
)

fast_info keys: ['currency', 'dayHigh', 'dayLow', 'exchange', 'fiftyDayAverage', 'lastPrice', 'lastVolume', 'marketCap', 'open', 'previousClose', 'quoteType', 'regularMarketPreviousClose', 'shares', 'tenDayAverageVolume', 'threeMonthAverageVolume', 'timezone', 'twoHundredDayAverage', 'yearChange', 'yearHigh', 'yearLow']
fast_info quote: {'lastPrice': 214.72000122070312, 'previousClose': 217.05, 'dayHigh': 218.74000549316406, 'dayLow': 214.5}


In [ ]:
import math
from datetime import date

DELTA_MAX = 0.21       # keep calls with delta < this
IV_FALLBACK = 0.20     # used when a contract has no IV in the chain


def _ncdf(x: float) -> float:
    """Standard normal CDF via math.erfc — no scipy needed."""
    return 0.5 * math.erfc(-x / math.sqrt(2))


def _bs_delta_call(S: float, K: float, T: float, sigma: float, r: float = 0.05) -> float:
    """Black-Scholes delta for a European call."""
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return 0.0
    d1 = (math.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    return _ncdf(d1)


# --- underlying price via yfinance (use price symbol, not options symbol) ---
_uf = yf.Ticker(_yf_price_symbol)
_last = getattr(_uf.fast_info, "last_price", None)
if not _last:
    _hist = _uf.history(period="5d")["Close"].dropna()
    if _hist.empty:
        raise RuntimeError(f"Could not fetch underlying price for {_yf_price_symbol}.")
    _last = float(_hist.iloc[-1])
underlying_price = float(_last)
print(f"underlying_price ({_yf_price_symbol}): {underlying_price}")

# --- fetch all contracts for the selected expiration ---
_page_results: list = []
_next_url: str | None = None
while True:
    if _next_url:
        _r = requests.get(_next_url, params={"apiKey": POLYGON_API_KEY}, timeout=30)
        _r.raise_for_status()
        _page = _r.json()
    else:
        _page = polygon_get(
            "/v3/reference/options/contracts",
            underlying_ticker=UNDERLYING,
            expiration_date=selected_expiration,
            contract_type="call",
            order="asc",
            sort="strike_price",
            limit=1000,
        )
    _page_results.extend(_page.get("results", []))
    _next_url = _page.get("next_url")
    if not _next_url:
        break

if not _page_results:
    raise RuntimeError("No contracts returned for the selected expiration.")

# --- compute delta for each contract ---
_dte = (date.fromisoformat(selected_expiration) - date.today()).days
T = max(_dte, 1) / 365.0

_rows = []
for c in _page_results:
    K = float(c["strike_price"])
    iv = float(c.get("implied_volatility") or IV_FALLBACK)
    delta = _bs_delta_call(underlying_price, K, T, iv)
    _rows.append({
        "ticker":          c.get("ticker"),
        "expiration_date": c.get("expiration_date"),
        "contract_type":   c.get("contract_type"),
        "exercise_style":  c.get("exercise_style"),
        "strike_price":    K,
        "iv_used":         round(iv, 4),
        "delta":           round(delta, 4),
    })

_all_contracts = pd.DataFrame(_rows)

# Filter: delta < DELTA_MAX
polygon_contracts = (
    _all_contracts[_all_contracts["delta"] < DELTA_MAX]
    .reset_index(drop=True)
)

print(f"{len(_all_contracts)} total calls → {len(polygon_contracts)} with delta < {DELTA_MAX}")
polygon_contracts[["ticker", "strike_price", "delta", "iv_used", "exercise_style"]]


underlying_price (^GSPC): 7230.1201171875
246 total calls → 40 with delta < 0.21


,ticker,strike_price,delta,iv_used,exercise_style
0,O:SPXW260507C07360000,7360.0,0.2079,0.2,european
1,O:SPXW260507C07365000,7365.0,0.1987,0.2,european
2,O:SPXW260507C07370000,7370.0,0.1898,0.2,european
3,O:SPXW260507C07375000,7375.0,0.1812,0.2,european
4,O:SPXW260507C07380000,7380.0,0.1728,0.2,european
5,O:SPXW260507C07385000,7385.0,0.1646,0.2,european
6,O:SPXW260507C07390000,7390.0,0.1567,0.2,european
7,O:SPXW260507C07395000,7395.0,0.1491,0.2,european
8,O:SPXW260507C07400000,7400.0,0.1417,0.2,european
9,O:SPXW260507C07405000,7405.0,0.1346,0.2,european


In [ ]:
# Enrich with bid/ask from Yahoo Finance (Polygon reference endpoint has no quotes)
_yf_ticker = yf.Ticker(_yf_symbol)
try:
    _yf_chain = _yf_ticker.option_chain(selected_expiration)
    _yf_calls = _yf_chain.calls[["strike", "bid", "ask", "lastPrice", "impliedVolatility", "openInterest"]].copy()
    _yf_calls = _yf_calls.rename(columns={
        "strike":           "strike_price",
        "lastPrice":        "last_price",
        "impliedVolatility":"iv_yahoo",
        "openInterest":     "open_interest",
    })
    polygon_contracts = polygon_contracts.merge(_yf_calls, on="strike_price", how="left")
    print(f"merged Yahoo bid/ask for {selected_expiration}: {_yf_calls['strike_price'].nunique()} strikes available")
except Exception as exc:
    print(f"Yahoo chain fetch failed ({exc}); bid/ask columns will be missing")

cols = ["ticker", "strike_price", "delta", "iv_used", "bid", "ask", "last_price", "open_interest"]
polygon_contracts[[c for c in cols if c in polygon_contracts.columns]]


Yahoo chain fetch failed (Expiration `2026-05-07` cannot be found. Available expirations are: []); bid/ask columns will be missing


,ticker,strike_price,delta,iv_used
0,O:SPXW260507C07360000,7360.0,0.2079,0.2
1,O:SPXW260507C07365000,7365.0,0.1987,0.2
2,O:SPXW260507C07370000,7370.0,0.1898,0.2
3,O:SPXW260507C07375000,7375.0,0.1812,0.2
4,O:SPXW260507C07380000,7380.0,0.1728,0.2
5,O:SPXW260507C07385000,7385.0,0.1646,0.2
6,O:SPXW260507C07390000,7390.0,0.1567,0.2
7,O:SPXW260507C07395000,7395.0,0.1491,0.2
8,O:SPXW260507C07400000,7400.0,0.1417,0.2
9,O:SPXW260507C07405000,7405.0,0.1346,0.2


## Polygon: fetch a snapshot for one option contract

This uses Polygon's `GET /v3/snapshot/options/{underlyingAsset}/{optionContract}` endpoint.


In [ ]:
polygon_contract_symbol = polygon_contracts.iloc[0]["ticker"]
polygon_snapshot = polygon_get(f"/v3/snapshot/options/{UNDERLYING}/{polygon_contract_symbol}")
polygon_snapshot


In [ ]:
snapshot_results = polygon_snapshot.get("results", {})
snapshot_summary = {
    "ticker": snapshot_results.get("details", {}).get("ticker"),
    "expiration_date": snapshot_results.get("details", {}).get("expiration_date"),
    "strike_price": snapshot_results.get("details", {}).get("strike_price"),
    "contract_type": snapshot_results.get("details", {}).get("contract_type"),
    "break_even_price": snapshot_results.get("break_even_price"),
    "implied_volatility": snapshot_results.get("implied_volatility"),
    "open_interest": snapshot_results.get("open_interest"),
    "delta": snapshot_results.get("greeks", {}).get("delta"),
    "gamma": snapshot_results.get("greeks", {}).get("gamma"),
    "theta": snapshot_results.get("greeks", {}).get("theta"),
    "vega": snapshot_results.get("greeks", {}).get("vega"),
    "last_quote_bid": snapshot_results.get("last_quote", {}).get("bid"),
    "last_quote_ask": snapshot_results.get("last_quote", {}).get("ask"),
    "underlying_price": snapshot_results.get("underlying_asset", {}).get("price"),
}

pd.DataFrame([snapshot_summary])


## Quick comparison


In [ ]:
comparison_df = pd.DataFrame([
    {
        "provider": "Yahoo",
        "contract": yahoo_contract_symbol,
        "expiration": selected_expiration,
        "strike": sample_call.get("strike"),
        "last_price": sample_call.get("lastPrice"),
        "bid": sample_call.get("bid"),
        "ask": sample_call.get("ask"),
        "implied_volatility": sample_call.get("impliedVolatility"),
        "open_interest": sample_call.get("openInterest"),
    },
    {
        "provider": "Polygon",
        "contract": snapshot_summary.get("ticker"),
        "expiration": snapshot_summary.get("expiration_date"),
        "strike": snapshot_summary.get("strike_price"),
        "last_price": None,
        "bid": snapshot_summary.get("last_quote_bid"),
        "ask": snapshot_summary.get("last_quote_ask"),
        "implied_volatility": snapshot_summary.get("implied_volatility"),
        "open_interest": snapshot_summary.get("open_interest"),
    },
])

comparison_df


## Source references

- yfinance docs: `Ticker.options` and `Ticker.option_chain(...)`
- Polygon contracts endpoint: `GET /v3/reference/options/contracts`
- Polygon contract snapshot endpoint: `GET /v3/snapshot/options/{underlyingAsset}/{optionContract}`
